# MineArt Diffusion: Cloud GPU Training (Tesla T4)
This notebook trains the MineArt diffusion model on a **Tesla T4 GPU (16GB VRAM)** using `batch_size=128`.

### Pre-requisites:
1. Go to **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU -> Save**.
2. Upload `data_64x64.zip`, `metadata_multicrop.csv`, `diffusion.py`, and optionally `best_diffusion.pt` to your Google Drive or the Colab files pane.

In [ ]:
# 1. Verify Tesla T4 GPU
!nvidia-smi

In [ ]:
# 2. Mount Google Drive (Recommended for easy saving & loading)
from google.colab import drive
drive.mount('/content/drive')

# Create project directories
!mkdir -p data/processed_64x64 checkpoints output_samples

In [ ]:
# 3. Extract Dataset & Copy Script from Drive
# Adjust path below if your zip file is in a subfolder in Google Drive
DRIVE_FOLDER = "/content/drive/MyDrive"

!unzip -q "{DRIVE_FOLDER}/data_64x64.zip" -d data/processed_64x64/
!cp "{DRIVE_FOLDER}/metadata_multicrop.csv" data/
!cp "{DRIVE_FOLDER}/diffusion.py" .

# If you uploaded the previous checkpoint to resume:
!cp "{DRIVE_FOLDER}/best_diffusion.pt" checkpoints/ 2>/dev/null || true

!echo "Dataset extracted: $(ls data/processed_64x64 | wc -l) images."

In [ ]:
# 4. Launch High-Speed Training (Batch Size 128 on Tesla T4)
# Runs 30 epochs (543 steps/epoch = 16,290 high-capacity steps, ~1 hour total)
!python diffusion.py \
    --train \
    --metadata data/metadata_multicrop.csv \
    --images data/processed_64x64 \
    --resume checkpoints/best_diffusion.pt \
    --epochs 30 \
    --batch-size 128 \
    --lr 0.0004

In [ ]:
# 5. Save the Newly Trained Model back to Google Drive
!cp checkpoints/best_diffusion.pt "{DRIVE_FOLDER}/best_diffusion_t4_trained.pt"
print("Trained model saved to Google Drive!")

In [ ]:
# 6. Generate Sample Artwork in Colab
from IPython.display import Image as IPImage, display

!python diffusion.py \
    --prompt "minecraft sunset mountains landscape pixel art" \
    --steps 50 \
    --checkpoint checkpoints/best_diffusion.pt \
    --output output_samples/cloud_painting.png

display(IPImage('output_samples/cloud_painting.png'))